<a href="https://colab.research.google.com/github/laramalkawi81-ops/DS230-Instacart-Project/blob/main/Copy_of_06_taskB_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 Model Comparison – Task B (Regression)

In this notebook, we compare multiple regression models for predicting the
number of days until the next order.  
All models are trained on the same sampled dataset to ensure fair comparison.

The goal is to evaluate performance using multiple metrics and select
the best-performing model.


In [3]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb


from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [4]:
DATA_PATH = '/content/drive/MyDrive/instacart_data/instacart_data'
data = pd.read_csv(f"{DATA_PATH}/model_data_sample.csv")

print(data.shape)
data.head()


(6550242, 19)


,order_id,product_id,add_to_cart_order,reordered,product_freq,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,days_since_prior_order_scaled,total_orders,mean_days_between_orders,product_reorder_rate,product_purchase_count,user_product_count,user_product_reorder_rate,order_hour
0,6,40462,1,0,0.000009,22352,prior,4,1,12,30.0,2.100730,9,21.777778,0.307692,52,1,0.000000,12
1,6,15873,2,0,0.000002,22352,prior,4,1,12,30.0,2.100730,9,21.777778,0.294118,17,1,0.000000,12
2,6,41897,3,0,0.000001,22352,prior,4,1,12,30.0,2.100730,9,21.777778,0.100000,10,1,0.000000,12
3,28,35108,1,0,0.000591,98256,prior,29,3,13,6.0,-0.477496,81,4.456790,0.609198,3892,3,0.666667,13
4,28,40593,2,1,0.000215,98256,prior,29,3,13,6.0,-0.477496,81,4.456790,0.463506,1603,2,0.500000,13


Features and Target

In [6]:
target = 'days_since_prior_order'
features = [
    'total_orders',
    'mean_days_between_orders',
    'product_reorder_rate',
    'product_purchase_count',
    'user_product_count',
    'user_product_reorder_rate',
    'order_dow',
    'order_hour'
]

X = data[features]
y = data[target]


In [7]:
train_data = data[data['order_number'] < data['order_number'].quantile(0.8)]
test_data  = data[data['order_number'] >= data['order_number'].quantile(0.8)]

X_train = train_data[features]
y_train = train_data[target]

X_test = test_data[features]
y_test = test_data[target]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (5238960, 8)
Test shape: (1311282, 8)


Feature Scaling

In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


 Linear Regression + Lasso + Ridge + ElasticNet


In [9]:
# Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

# Lasso
lasso = Lasso(alpha=0.01)
lasso.fit(X_train_scaled, y_train)
y_pred_lasso = lasso.predict(X_test_scaled)

# Ridge
ridge = Ridge(alpha=1)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)

# ElasticNet
en = ElasticNet(alpha=0.01, l1_ratio=0.5)
en.fit(X_train_scaled, y_train)
y_pred_en = en.predict(X_test_scaled)


 Decision Tree + Random Forest


In [10]:
# Decision Tree
dt = DecisionTreeRegressor(max_depth=10, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

# Random Forest
rf = RandomForestRegressor(n_estimators=10, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)


K-Nearest Neighbors + SVR


In [ ]:
# KNN
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)

# SVR (Linear)
svr_linear = SVR(kernel='linear')
svr_linear.fit(X_train_scaled, y_train)
y_pred_svr_linear = svr_linear.predict(X_test_scaled)

# SVR (RBF)
svr_rbf = SVR(kernel='rbf')
svr_rbf.fit(X_train_scaled, y_train)
y_pred_svr_rbf = svr_rbf.predict(X_test_scaled)


LightGBM


In [ ]:
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_test  = lgb.Dataset(X_test, label=y_test, reference=lgb_train)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'verbosity': -1,
    'seed': 42
}

gbm = lgb.train(params, lgb_train, num_boost_round=50)
y_pred_gbm = gbm.predict(X_test)


 Evaluation Metrics


In [ ]:
models_pred = {
    "Linear Regression": y_pred_lr,
    "Lasso": y_pred_lasso,
    "Ridge": y_pred_ridge,
    "ElasticNet": y_pred_en,
    "Decision Tree": y_pred_dt,
    "Random Forest": y_pred_rf,
    "KNN": y_pred_knn,
    "SVR Linear": y_pred_svr_linear,
    "SVR RBF": y_pred_svr_rbf,
    "LightGBM": y_pred_gbm
}

for name, preds in models_pred.items():
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)
    print(f"{name} -> MAE: {mae:.3f}, RMSE: {rmse:.3f}, R²: {r2:.3f}")


 Scatter Plot: Actual vs Predicted


In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred_rf, alpha=0.3)
plt.plot([0,30], [0,30], linestyle='--', color='red')
plt.xlabel("Actual Days")
plt.ylabel("Predicted Days (Random Forest)")
plt.title("Actual vs Predicted Days Until Next Order")
plt.show()
